# DIESEL — Notebook 02: Round 1 QLoRA Training
Fine-tune Llama-3.1-8B-Instruct with QLoRA on Spider.

**Runtime:** Select the **L4 GPU** kernel via the Colab extension.

In [12]:
%cd /content
!rm -rf /content/Text-to-SQL-LLM
!git clone https://github.com/Deepak-Lingala/Text-to-SQL-LLM.git /content/Text-to-SQL-LLM
%cd /content/Text-to-SQL-LLM


/content
Cloning into '/content/Text-to-SQL-LLM'...
remote: Enumerating objects: 56, done.
remote: Counting objects: 100% (56/56), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 56 (delta 26), reused 52 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (56/56), 229.96 KiB | 32.85 MiB/s, done.
Resolving deltas: 100% (26/26), done.
/content/Text-to-SQL-LLM


In [13]:
# Install dependencies
!pip install -q torch transformers peft trl bitsandbytes
!pip install -q accelerate datasets sqlparse sqlglot
!pip install -q wandb matplotlib seaborn scipy scikit-learn python-dotenv tqdm

In [14]:
# HuggingFace login (required for gated Llama model)
from huggingface_hub import login
login()  # Will prompt for token, or use: login(token='hf_YOUR_TOKEN')

In [15]:
# === Colab Setup: Clone project & set path ===
import os, sys

if not os.path.exists('/content/Text-to-SQL-LLM/src'):
    !git clone https://github.com/Deepak-Lingala/Text-to-SQL-LLM.git /content/Text-to-SQL-LLM

os.chdir('/content/Text-to-SQL-LLM')
sys.path.insert(0, '/content/Text-to-SQL-LLM')
print('Project root:', os.getcwd())

import torch
import gc

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

Project root: /content/Text-to-SQL-LLM
CUDA available: True
GPU: NVIDIA L4
VRAM: 23.7 GB


In [16]:
import os, sys

# Ensure current directory is the project root
os.chdir('/content/Text-to-SQL-LLM')

# Ensure project root is at the very beginning of sys.path for imports
project_root = '/content/Text-to-SQL-LLM'
if project_root in sys.path:
    sys.path.remove(project_root)
sys.path.insert(0, project_root)

print('Current working directory:', os.getcwd())
print('Python path:', sys.path)

from src.config import get_config, TrainingConfig
from dataclasses import replace

config = get_config()

# ============================================================
# SET DRY_RUN = True for a quick smoke test (10 steps, ~2 min)
# SET DRY_RUN = False for full training (~2-3 hours on L4)
# ============================================================
DRY_RUN = True

if DRY_RUN:
    config = replace(
        config,
        training=replace(
            config.training,
            max_steps=10,
            logging_steps=1,
            eval_steps=10,
            save_steps=10,
            report_to='none',
        ),
    )
    print('*** DRY RUN MODE: 10 steps ***')
else:
    print('*** FULL TRAINING MODE ***')

print(f'Model: {config.model.model_name}')
print(f'LoRA rank: {config.lora.r}, alpha: {config.lora.lora_alpha}')
print(f'LR: {config.training.learning_rate}')
print(f'Epochs: {config.training.num_train_epochs}')

Current working directory: /content/Text-to-SQL-LLM
Python path: ['/content/Text-to-SQL-LLM', '/content/Text-to-SQL-LLM', '/content', '/env/python', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.12/dist-packages/IPython/extensions', '/root/.ipython']
*** DRY RUN MODE: 10 steps ***
Model: meta-llama/Llama-3.1-8B-Instruct
LoRA rank: 16, alpha: 32
LR: 0.0002
Epochs: 3


## Load & Prepare Data

In [17]:
from src.data_loader import load_spider_dataset, prepare_training_data

print('Loading Spider dataset...')
dataset, schema_manager = load_spider_dataset(config)

print('\nFormatting training prompts...')
train_data = prepare_training_data(dataset, schema_manager, config, split='train')
eval_data = prepare_training_data(dataset, schema_manager, config, split='validation')

print(f'\nTrain: {len(train_data)} examples')
print(f'Eval:  {len(eval_data)} examples')
print(f'\nSample prompt (first 300 chars):')
print(train_data[0]['text'][:300] + '...')

Loading Spider dataset...
Loading Spider dataset from HuggingFace...


Generating train split:   0%|          | 0/7000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1034 [00:00<?, ? examples/s]

  Loaded 160 database schemas
  Train: 7000 examples
  Validation: 1034 examples

Formatting training prompts...


Formatting validation: 100%|██████████| 1034/1034 [00:00<00:00, 8260.61it/s]


Train: 7000 examples
Eval:  1034 examples

Sample prompt (first 300 chars):
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an expert SQL assistant. Given a database schema and a natural language question, generate the correct SQL query. Output ONLY the SQL query, nothing else.<|eot_id|><|start_header_id|>user<|end_header_id|>

Given the following datab...


## Train

In [18]:
from src.train import train

trainer, result = train(
    train_dataset=train_data,
    eval_dataset=eval_data,
    config=config,
    output_dir=config.paths.round1_dir,
    run_name='diesel_round1',
)

DIESEL Training Pipeline
Loading base model: meta-llama/Llama-3.1-8B-Instruct
  Quantization: 4-bit nf4
  Compute dtype: float16


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

  Model loaded on: cuda:0
  Trainable parameters before LoRA: 1,050,939,392


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.



  LoRA Configuration:
    Rank (r): 16
    Alpha: 32
    Target modules: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
    Trainable params: 41,943,040 / 4,582,543,360 (0.92%)


...

Truncating eval dataset:   0%|          | 0/1034 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.



===================Training Configuration===================
  Model: meta-llama/Llama-3.1-8B-Instruct
  LoRA rank: 16, alpha: 32
  Learning rate: 0.0002
  Epochs: 3
  Effective batch size: 16
  Max seq length: 1024
  Train examples: 7000
  Eval examples: 1034
  Output dir: outputs/round1


Step,Training Loss,Validation Loss
10,0.799635,0.858321


***** train metrics *****
  total_flos               =   962688GF
  train_loss               =     1.3799
  train_runtime            = 0:03:03.90
  train_samples_per_second =       0.87
  train_steps_per_second   =      0.054

=====================Training Complete======================
  Final adapter saved to: outputs/round1/final_adapter
  Train loss: 1.3799
  Train runtime: 183.9s


In [ ]:
# Cleanup GPU memory
del trainer
gc.collect()
torch.cuda.empty_cache()

print('\n' + '=' * 60)
print('Round 1 Training Complete!')
print('=' * 60)
adapter_path = os.path.join(config.paths.round1_dir, 'final_adapter')
print(f'Adapter saved to: {adapter_path}')
print(f'Loss: {result.metrics.get("train_loss", "N/A")}')
print(f'Runtime: {result.metrics.get("train_runtime", 0):.0f}s')
print(f'\nNext: Run 03_error_analysis.ipynb')